homework 1.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
m=5 #质量
c=1 #阻尼
k=10 #弹性系数
x_0=0.1 #初始位移
v_0=0 #初始速度
u_0=0 #初始位移
t=np.linspace(0,100,1000) #时间范围
omega=np.sqrt(k/m) #自然频率
zeta=c/(2*np.sqrt(m*k)) #阻尼比
omega_d=omega*np.sqrt(1-zeta**2) #阻尼频率
u=np.exp(-zeta*omega*t)*(x_0*np.cos(omega_d*t)+((v_0+zeta*omega*x_0)/omega_d)*np.sin(omega_d*t)) #位移响应
plt.rcParams['font.sans-serif']=['DejaVu Sans'] #设置字体
plt.plot(t,u)
plt.title('Vibration Response')
plt.xlabel('time (s)')
plt.ylabel('displacement (m)')
plt.grid()
plt.show()

homework 1.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
m=5 #质量
c=1 #阻尼
k=10 #弹性系数
t=np.linspace(0,100,1000) #时间范围
f=10*np.sin(2*t) #外部激励
A=np.array([[0,1],[-k/m,-c/m]]) #A矩阵
B=np.array([[0],[1/m]]) #B矩阵
C=np.array([[1,0]]) #C矩阵
D=np.array([[0]]) #D矩阵
sys=signal.StateSpace(A,B,C,D) #状态空间模型
t,y,x=signal.lsim(sys,f,t) #求解系统响应
fig,ax=plt.subplots()
plt.rcParams['font.sans-serif']=['SimHei'] #设置中文字体
plt.rcParams['axes.unicode_minus']=False #设置负号显示
ax.set_title('稳态响应')
ax.set_xlabel('时间 (s)')
ax.set_ylabel('位移 (m)')
ax.plot(t,y)
plt.show()

homework 2.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

N=2000 #采样点数
T=1 #采样时长
fs=N/T #采样频率
df=fs/N #频率分辨率
t=np.linspace(0,T,N) #采样时间点
x=1.5*np.sin(2*np.pi*151*t)+np.sin(2*np.pi*501*t)+np.random.randn(N) #采样信号

window=np.hanning(N) #汉宁窗
sf2=N/np.sum(window**2) #窗能量修正因子
x_windowed=x*window #加窗
y=np.fft.fft(x_windowed/N) #FFT变换
P=abs(y)**2/df*2*sf2 #功率谱密度
f=np.fft.fftfreq(N,d=1/fs) #双边频率
idx=f>=0 #转换为单边谱
f_idx=f[idx]
P_idx=P[idx]

nperseg=N//8 #窗口长度
noverlap=N//16 #重叠长度
f_welch,P_welch=signal.welch(x,fs=fs,window='hann',nperseg=nperseg,noverlap=noverlap,return_onesided=True) #welch法

fig,ax=plt.subplots(2,1,figsize=(10,10))

plt.rcParams['font.sans-serif']=['DejaVu Sans'] #设置字体
plt.rcParams['axes.unicode_minus']=False #设置负号显示
ax[0].set_title('Modified Periodogram (Hanning window)')
ax[0].set_xlabel('Frequency (Hz)')
ax[0].set_ylabel('PSD (V^2/Hz)')
ax[0].plot(f_idx,P_idx)

ax[1].set_title(f'Welch Method (nperseg={nperseg}, noverlap={noverlap})')
ax[1].set_xlabel('Frequency (Hz)')
ax[1].set_ylabel('PSD (V^2/Hz)')
ax[1].semilogy(f_welch,P_welch)

homework 3.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh, solve
L_t=1 #梁长度/m
N=5 #单元数
E=71e9 #弹性模量/GPa
Rho=2.77e3 #质量密度/(kg/m^3)
b=0.1 #横截面宽度/m
h=0.01 #横截面高度/m
A=b*h #截面积/m^2
I=b*h**3/12 #转动惯量/m^4

alpha=0.01
beta=1e-4
L=L_t/N
k_e=E*I/L**3*np.array([[12, 6*L, -12, 6*L],
                      [6*L, 4*L**2, -6*L, 2*L**2],
                      [-12, -6*L, 12, -6*L],
                      [6*L, 2*L**2, -6*L, 4*L**2]])

m_e=Rho*A*L/420*np.array([[156, 22*L, 54, -13*L],
                         [22*L, 4*L**2, 13*L, -3*L**2],
                         [54, 13*L, 156, -22*L],
                         [-13*L, -3*L**2, -22*L, 4*L**2]])

c_e=alpha*m_e+beta*k_e

def assemble(N,k_e,m_e,c_e):
    nnode=N+1
    ndof=2*nnode
    K=np.zeros((ndof,ndof))
    M=np.zeros((ndof,ndof))
    for i in range(N):
        dof=[2*i, 2*i+1, 2*i+2, 2*i+3]
        for j in range(4):
            for k in range(4):
                K[dof[j], dof[k]] += k_e[j,k]
                M[dof[j], dof[k]] += m_e[j,k]
    return K, M

def boundary_condition(K, M, fixed_dofs):
    ndof=K.shape[0]
    free_dofs=[i for i in range(ndof) if i not in fixed_dofs]
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    M_ff = M[np.ix_(free_dofs, free_dofs)]
    return K_ff, M_ff, free_dofs

K, M=assemble(N, k_e, m_e, c_e)

fixed_dots=[0,1]
K_ff, M_ff, free_dofs=boundary_condition(K, M, fixed_dots)

eigvals, eigvecs_ff=eigh(K_ff,M_ff)
omega_n=np.sqrt(eigvals)
f_n=omega_n/(2*np.pi)

nmodes=eigvecs_ff.shape[1]
ndof_total=K.shape[0]
eigvecs_full=np.zeros((ndof_total, nmodes))
for i, dof in enumerate(free_dofs):
    eigvecs_full[dof, :]=eigvecs_ff[i, :]

nnode = N + 1
x = np.linspace(0, L, nnode)
plt.figure(figsize=(12, 8))
for i in range(min(nmodes, len(f_n))):
    
    w = eigvecs_full[0:2*nnode:2, i]
   
    w = w / np.max(np.abs(w))
    
    plt.subplot(5, 2, i+1)
    plt.plot(x, w, 'b-o', linewidth=2, markersize=6)
    plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    plt.xlabel('Position (m)', fontsize=10)
    plt.ylabel('Normalized deflection', fontsize=10)
    plt.title(f'Mode{i+1} (f = {f_n[i]:.2f} Hz)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xlim([0, L])
    
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh, solve
L_t=1 #梁长度/m
N=5 #单元数
E=71e9 #弹性模量/GPa
Rho=2.77e3 #质量密度/(kg/m^3)
b=0.1 #横截面宽度/m
h=0.01 #横截面高度/m
A=b*h #截面积/m^2
I=b*h**3/12 #转动惯量/m^4

alpha=0.01
beta=1e-4
L=L_t/N
k_e=E*I/L**3*np.array([[12, 6*L, -12, 6*L],
                      [6*L, 4*L**2, -6*L, 2*L**2],
                      [-12, -6*L, 12, -6*L],
                      [6*L, 2*L**2, -6*L, 4*L**2]])

m_e=Rho*A*L/420*np.array([[156, 22*L, 54, -13*L],
                         [22*L, 4*L**2, 13*L, -3*L**2],
                         [54, 13*L, 156, -22*L],
                         [-13*L, -3*L**2, -22*L, 4*L**2]])

c_e=alpha*m_e+beta*k_e

def assemble(N,k_e,m_e,c_e):
    nnode=N+1
    ndof=2*nnode
    K=np.zeros((ndof,ndof))
    M=np.zeros((ndof,ndof))
    for i in range(N):
        dof=[2*i, 2*i+1, 2*i+2, 2*i+3]
        for j in range(4):
            for k in range(4):
                K[dof[j], dof[k]] += k_e[j,k]
                M[dof[j], dof[k]] += m_e[j,k]
    return K, M

def boundary_condition(K, M, fixed_dofs):
    ndof=K.shape[0]
    free_dofs=[i for i in range(ndof) if i not in fixed_dofs]
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    M_ff = M[np.ix_(free_dofs, free_dofs)]
    return K_ff, M_ff, free_dofs

K, M=assemble(N, k_e, m_e, c_e)

fixed_dots=[0,1]
K_ff, M_ff, free_dofs=boundary_condition(K, M, fixed_dots)

eigvals, eigvecs_ff=eigh(K_ff,M_ff)
omega_n=np.sqrt(eigvals)
f_n=omega_n/(2*np.pi)

for i in range(len(omega_n)):
    m_modal=eigvecs_ff[:, i] @ M_ff @eigvecs_ff[:, i]
    eigvecs_ff[:, i]=eigvecs_ff[:, i]/np.sqrt(m_modal)
nmodes=5
omega_n=omega_n[:nmodes]
eigvecs_ff=eigvecs_ff[:, :nmodes]

zeta_n=[]
for i in range(nmodes):
    zeta=(alpha/(2*omega_n[i])+beta*omega_n[i]/2)
    zeta_n.append(zeta)

force_dofs = [2, 4]
amplitudes=[10.0, 5.0]
target_dof = 6
F_indices=[]
for dof in force_dofs:
    if dof in free_dofs:
        F_indices.append(free_dofs.index(dof))

duration = 60.0         
dt = 0.005              
t = np.arange(0, duration, dt)

ndof=len(free_dofs)
F_physical=np.zeros((ndof,len(t)))
omega=[5, 8]


for idx, amp, ome in zip(F_indices, amplitudes, omega):
    F_physical[idx, :]=amp*np.sin(ome*t)

 
F_model=eigvecs_ff.T @ F_physical 
q=np.zeros((nmodes,len(t)))

for i in range(nmodes):
    for j, time in enumerate(t):
        if omega_n[i]>0:
            response_sum=0
            for idx, amp, ome in zip(F_indices, amplitudes, omega):
                lam=ome/omega_n[i]
                H=1/(omega_n[i]**2*((1-lam**2)**2+(2*zeta_n[i]*lam)**2))
                phase=np.arctan2(2*zeta_n[i]*lam, 1-lam**2)
                modal_force_amp = eigvecs_ff[idx, i] * amp
                response_sum += H * modal_force_amp * np.sin(ome*time - phase)
            q[i, j] = response_sum
u_physical=eigvecs_ff @ q


fig = plt.figure(figsize=(16, 12))


target_idx = free_dofs.index(target_dof)
response = u_physical[target_idx, :]

ax1 = plt.subplot(3, 3, 1)
ax1.plot(t, response*1000, 'b-', linewidth=0.8)
ax1.set_xlabel('Time (s)', fontsize=10)
ax1.set_ylabel('Displacement (mm)', fontsize=10)
ax1.set_title('DOF5 Time Response (0-20s)', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, duration])



homework 4.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ct

#======================
# 参数
#======================

M=np.array([[1,0],
            [0,1]])

K=np.array([[2,-1],
            [-1,2]])

C=0.01*K

#======================
# 建立状态空间模型
#======================

n=2

A=np.block([
    [np.zeros((n,n)),np.eye(n)],
    [-np.linalg.inv(M)@K,
     -np.linalg.inv(M)@C]
])

B=np.vstack([
    np.zeros((n,n)),
    np.linalg.inv(M)
])

Cout=np.hstack([
    np.eye(n),
    np.zeros((n,n))
])

D=np.zeros((n,n))

G=ct.ss(A,B,Cout,D)

#======================
# H∞设计
#======================

s=ct.tf([1,0],[1])

w0=12

# 避免TF直接幂运算问题
num=(s/(2**(1/3))+w0)
den=(s+(0.0001**(1/3))*w0)

w1=(num*num*num)/(den*den*den)
w2=ct.tf([0.1],[1])

Khinf,CL,gam=ct.mixsyn(
    G,
    w1,
    w2,
    None
)

print("γ=",gam)

#======================
# 灵敏度函数
#======================

I=ct.ss([],[],[],np.eye(2))

S=ct.feedback(I,G*Khinf)

sysd=S*G
sysu=-Khinf*S*G

#======================
# 输入扰动
#======================

t=np.arange(0,20,0.1)

Fd=np.zeros((2,len(t)))
Fd[0,:]=np.sin(2*t)

#======================
# 开环响应
#======================

T1,Y1=ct.forced_response(
    G,
    T=t,
    U=Fd
)

#======================
# 闭环响应
#======================

T2,Y2=ct.forced_response(
    sysd,
    T=t,
    U=Fd
)

#======================
# 控制力
#======================

Tu,Fu=ct.forced_response(
    sysu,
    T=t,
    U=Fd
)

#======================
# 联合作用
#======================

f=np.zeros((2,len(t)))

f[0,:]=Fd[0,:]+Fu[0,:]
f[1,:]=Fu[1,:]

T3,Y3=ct.forced_response(
    G,
    T=t,
    U=f
)

#======================
# 绘图
#======================

plt.figure()

plt.plot(T1,Y1[0,:],'r',label='Open-loop')

plt.plot(T2,Y2[0,:],'b',label='Closed-loop')

plt.plot(T3,Y3[0,:],'k--',
         label='Open+Control')

plt.xlabel("Time(s)")
plt.ylabel("Displacement")
plt.legend()
plt.grid()

plt.figure()

plt.plot(Tu,Fu[0,:],
         label='Control force')

plt.plot(t,Fd[0,:],
         'r',
         label='Disturbance')

plt.xlabel("Time(s)")
plt.ylabel("Force")
plt.legend()
plt.grid()

plt.show()